# 01 — Dataset profiling (MIMIC-IV Demo 2.2)

**KHÔNG PHẢI KẾT QUẢ NGHIÊN CỨU.** Theo mục **5.1–5.2**, kiểm tra số dòng, số mã bệnh nhân, khóa trùng, dữ liệu thiếu, lỗi kiểu số và khóa tham chiếu trước chọn subset.

Notebook mặc định đọc dữ liệu **SYNTHETIC do nhóm tự tạo** để chứng minh pipeline hoạt động; phải chuyển rõ `INPUT_MODE` sang `real` khi có CSV thật.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Notebook must run from this repository folder or one of its subfolders")
SYNTHETIC = ROOT / "data" / "sample_synthetic"
print("Repository:", ROOT)
from analysis_core.dataset import load_csv, profile_mimic_like

# CHỌN NGUỒN DỮ LIỆU MỘT CÁCH TƯỜNG MINH; KHÔNG TỰ ĐỘNG FALLBACK.
INPUT_MODE = "real"  # đổi thành "real" sau khi tải bộ CSV thật
REAL_FILES = {
    "patients": ROOT / "data" / "raw" / "patients.csv.gz",       # chỉnh đúng đường dẫn
    "labevents": ROOT / "data" / "raw" / "labevents.csv.gz",   # chỉnh đúng đường dẫn
    "d_labitems": ROOT / "data" / "raw" / "d_labitems.csv.gz" # chỉnh đúng đường dẫn
}
if INPUT_MODE not in {"synthetic", "real"}:
    raise ValueError("INPUT_MODE must be 'synthetic' or 'real'")
if INPUT_MODE == "synthetic":
    print("⚠️ SYNTHETIC FIXTURE ONLY — DO NOT REPORT AS MIMIC OR PROTOTYPE RESULT")
    paths = {name: SYNTHETIC / f"{name}.csv" for name in REAL_FILES}
else:
    print("REAL CSV INPUT — verify dataset license, version 2.2, hash and paths")
    paths = REAL_FILES

patients, labevents, d_labitems = [load_csv(paths[name]) for name in ("patients", "labevents", "d_labitems")]
print("Loaded:", {name: str(path) for name, path in paths.items()})


Repository: /home/thien408/Projects/2026-2027/dacn/lis-fhir-emr-stat-audit
REAL CSV INPUT — verify dataset license, version 2.2, hash and paths
Loaded: {'patients': '/home/thien408/Projects/2026-2027/dacn/lis-fhir-emr-stat-audit/data/raw/patients.csv.gz', 'labevents': '/home/thien408/Projects/2026-2027/dacn/lis-fhir-emr-stat-audit/data/raw/labevents.csv.gz', 'd_labitems': '/home/thien408/Projects/2026-2027/dacn/lis-fhir-emr-stat-audit/data/raw/d_labitems.csv.gz'}


## A. Số bản ghi, số lượng định danh

`rows` là số bản ghi trong bảng, `unique_subject_id` chỉ tính trong các bảng có cột đó; không suy `n` được nạp vào LIS từ số lượng public dataset.

In [2]:
summary, missing, issues = profile_mimic_like(patients, labevents, d_labitems)
display(summary)


,table,rows,unique_subject_id,unique_itemid
0,patients,100,100.0,NaN
1,labevents,107727,100.0,498.0
2,d_labitems,1622,NaN,1622.0


## B. Missing values theo từng cột

In [3]:
display(missing.sort_values(["table", "missing_count"], ascending=[True, False]).reset_index(drop=True))


,table,column,missing_count,total_rows
0,d_labitems,label,3,1622
1,d_labitems,itemid,0,1622
2,d_labitems,fluid,0,1622
3,d_labitems,category,0,1622
4,labevents,order_provider_id,90897,107727
5,labevents,comments,89273,107727
6,labevents,flag,67452,107727
7,labevents,hadm_id,28420,107727
8,labevents,ref_range_lower,18728,107727
9,labevents,ref_range_upper,18728,107727


## C. Dữ liệu trùng, khóa tham chiếu không khớp, lỗi kiểu số

Không `dropna`/`merge(inner)` âm thầm trước khi tổng kết. Nếu cần loại, mỗi bản ghi bị loại phải có lý do và số lượng riêng.

In [4]:
display(issues)
display(labevents.groupby("itemid", dropna=False).size().rename("records").reset_index().sort_values("records", ascending=False))


,check,count
0,duplicate_patient_subject_id,0
1,duplicate_labevent_id,0
2,duplicate_d_labitems_itemid,0
3,labevents_patient_not_found,0
4,labevents_itemid_not_found,0
5,labevents_invalid_nonempty_valuenum,0


,itemid,records
121,50971,3022
129,50983,3007
81,50912,3003
71,50902,2981
144,51006,2974
...,...,...
174,51080,1
178,51086,1
492,52419,1
493,52425,1


## D. Nhật ký kiểm chứng thủ công

Trước khi thực nghiệm chính thức hãy ghi ở `run_manifest.json`: dataset version, URL, giấy phép, SHA-256, lựa chọn itemid, tiêu chí loại, mã commit, số bệnh nhân và số kết quả thực. Phần tiền xử lý phải tách **bản ghi bị loại** theo từng lý do. Tệp synthetic ở đây cố tình có lỗi để kiểm tra chức năng phát hiện.